In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.getActiveSession()

if spark:
    spark.stop()
    print("Stopped session, starting new session")
else:
    print("Starting Session")

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Starting Session


25/02/28 13:10:32 WARN Utils: Your hostname, Noahs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.210 instead (on interface en0)
25/02/28 13:10:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/28 13:10:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/02/28 13:10:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark.version

'3.5.4'

In [4]:
df = spark.read.parquet(
"/Users/noahbrannon/DEZoomcamp/Week5/code/yellow_tripdata_2024-10.parquet")

In [5]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double')]

In [6]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [7]:
df_repartition = df.repartition(4)



In [8]:
df_repartition.write.mode("overwrite").parquet("/Users/noahbrannon/DEZoomcamp/Week5/code/data/homework")

In [10]:
import os

versions = [f"{i}" for i in range(4)] 

for v in versions:
    file_path = f"/Users/noahbrannon/DEZoomcamp/Week5/code/data/homework/part-0000{v}-9929529c-4143-4990-a43b-e094783d22dc-c000.snappy.parquet"
    
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"File size of {file_path}: {file_size / (1024*1024):.2f} MB")
    else:
        print(f"File not found: {file_path}")



File size of /Users/noahbrannon/DEZoomcamp/Week5/code/data/homework/part-00000-9929529c-4143-4990-a43b-e094783d22dc-c000.snappy.parquet: 22.38 MB
File size of /Users/noahbrannon/DEZoomcamp/Week5/code/data/homework/part-00001-9929529c-4143-4990-a43b-e094783d22dc-c000.snappy.parquet: 22.40 MB
File size of /Users/noahbrannon/DEZoomcamp/Week5/code/data/homework/part-00002-9929529c-4143-4990-a43b-e094783d22dc-c000.snappy.parquet: 22.37 MB
File size of /Users/noahbrannon/DEZoomcamp/Week5/code/data/homework/part-00003-9929529c-4143-4990-a43b-e094783d22dc-c000.snappy.parquet: 22.43 MB


In [14]:
df.registerTempTable("yellow2024_10")

/Users/noahbrannon/DEZoomcamp/my_env/lib/python3.9/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [42]:
spark.sql("""
SELECT COUNT(*) 
FROM yellow2024_10 
WHERE tpep_pickup_datetime LIKE '2024-10-15%'
""").show()


+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [27]:
spark.sql("""
SELECT 
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS time_traveled
FROM yellow2024_10
ORDER BY time_traveled DESC
LIMIT 1
""").show()


+------------------+
|     time_traveled|
+------------------+
|162.61777777777777|
+------------------+



In [26]:
spark_ui_port = spark.sparkContext.uiWebUrl
spark_ui_port

'http://192.168.0.210:4041'

In [34]:
df_taxi = spark.read.option("header", "true").csv('/Users/noahbrannon/DEZoomcamp/Week5/code/taxi_zone_lookup.csv')

In [35]:
df_taxi.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [36]:
df_taxi.registerTempTable("taxi_lookup")

/Users/noahbrannon/DEZoomcamp/my_env/lib/python3.9/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [39]:
spark.sql("""
SELECT taxi_lookup.Zone, COUNT(*) AS trip_count
FROM yellow2024_10
JOIN taxi_lookup 
ON yellow2024_10.PULocationID = taxi_lookup.LocationID
GROUP BY taxi_lookup.Zone
ORDER BY trip_count
LIMIT 1
""").show()


+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+

